# SBA Loan Dataset Processing Summary

## 1. Dataset and Supporting Files

This notebook loads the SBA loan CSV and its raw data dictionary from the workspace staging directory under `data/raw/sba`.

- SBA dataset: `data/raw/sba/FOIA_7a_FY2020_Present_asof_260630.csv`
- Data dictionary: `data/raw/sba/data_dictionary_raw.csv`

The workflow removes exact duplicate rows, summarizes missing values, drops columns with more than 90% missing data, standardizes column names, converts date-like fields to datetime, cleans currency values, normalizes object columns, and optionally converts low-cardinality text columns to categorical dtype before a final review of the cleaned dataset.

## 2. Data Dictionary Validation

The raw data dictionary is loaded separately and checked against the SBA column names to identify which fields match the dictionary schema and which remain unmatched.

## 3. Final Output

The cleaned SBA dataset remains in the workspace staging area and is ready for downstream EDA, feature engineering, and modeling work.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

sba_data_path = workspace_root / "data" / "raw" / "sba" / "FOIA_7a_FY2020_Present_asof_260630.csv"
data_dict_path = workspace_root / "data" / "raw" / "sba" / "data_dictionary_raw.csv"

# Load SBA data and dictionary
df = pd.read_csv(sba_data_path, low_memory=False)
print("Initial shape:", df.shape)

data_dict = pd.read_csv(data_dict_path)
print("Dictionary shape:", data_dict.shape)
print(data_dict.head())

# Remove exact duplicate records
df = df.drop_duplicates().reset_index(drop=True)
print("After duplicate removal:", df.shape)

# Missing-value summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(df)) * 100
missing_df = pd.DataFrame({
    "missing_count": missing_summary,
    "missing_pct": missing_pct,
})
print(missing_df[missing_df["missing_count"] > 0].head(20))

# Drop columns with excessive missingness (>90%)
threshold = 0.90
cols_to_drop = missing_df[missing_df["missing_pct"] > threshold * 100].index.tolist()
print("Dropping columns:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

# Normalize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^\w]", "", regex=True)
)
print("Columns:", df.columns.tolist())

# Convert date-like columns
for col in [c for c in df.columns if "date" in c]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Clean numeric/currency-like fields
for col in [
    c for c in df.columns
    if df[c].dtype == object and df[c].astype(str).str.contains(r"\$", na=False).any()
]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("nan", np.nan)
        .astype(float)
    )

# Clean object columns
obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

# Optional category conversion for low-cardinality columns
categorical_candidates = [c for c in obj_cols if df[c].nunique(dropna=True) < 50]
for col in categorical_candidates:
    df[col] = df[col].astype("category")

# Compare dictionary fields with data columns
if not data_dict.empty:
    dict_key_col = data_dict.columns[0]
    matched_cols = [
        c for c in df.columns
        if c in data_dict[dict_key_col].astype(str).str.lower().str.strip().values
    ]
    unmatched_cols = [c for c in df.columns if c not in matched_cols]
    print("Matched columns:", matched_cols)
    print("Unmatched columns:", unmatched_cols)

print(df.shape)
df.info()
df.head()


Initial shape: (388338, 42)
Dictionary shape: (42, 2)
   Field Name                                         Definition
0    AsOfDate                    Date when the data was recorded
1     Program  Indicator of whether loan was approved under S...
2  LocationID                        SBA's unique lender ID code
3    BorrName                                      Borrower name
4  BorrStreet                            Borrower street address
After duplicate removal: (387947, 42)
                            missing_count  missing_pct
ChargeOffDate                      381059    98.224500
BankNCUANumber                     376814    97.130278
FranchiseName                      341569    88.045274
FranchiseCode                      341405    88.003000
PaidInFullDate                     319786    82.430332
SoldSecMrktInd                     271194    69.904910
FirstDisbursementDate               70901    18.275950
BankFDICNumber                      41301    10.646042
BusinessAge            

/tmp/ipykernel_38944/772490153.py:67: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns


Matched columns: ['asofdate', 'program', 'locationid', 'borrname', 'borrstreet', 'borrcity', 'borrstate', 'borrzip', 'bankname', 'bankfdicnumber', 'bankstreet', 'bankcity', 'bankstate', 'bankzip', 'grossapproval', 'sbaguaranteedapproval', 'approvaldate', 'approvalfy', 'firstdisbursementdate', 'processingmethod', 'initialinterestrate', 'fixedorvariableinterestind', 'terminmonths', 'naicscode', 'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice', 'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus', 'paidinfulldate', 'grosschargeoffamount', 'revolverstatus', 'jobssupported', 'collateralind', 'soldsecmrktind']
Unmatched columns: []
(387947, 40)
<class 'pandas.DataFrame'>
RangeIndex: 387947 entries, 0 to 387946
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   asofdate                    387947 non-null 

,asofdate,program,locationid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,congressionaldistrict,businesstype,businessage,loanstatus,paidinfulldate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,2026-06-30,7A,29805,"AMERIPRO CONSTRUCTION SERVICES, INC.",1403 SENTRY LANE,Norristown,PA,19403,"TD Bank, National Association",18409.0,...,4.0,PARTNERSHIP,Unanswered,P I F,2024-11-30,0.0,Y,5.0,N,NaN
1,2026-06-30,7A,84894,Meluota Corp,2702 ASTORIA BLVD,ASTORIA,NY,11102,"Santander Bank, National Association",29950.0,...,10.0,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,0.0,Y,3.0,N,NaN
2,2026-06-30,7A,53803,THOMAS W CHASE,1700 GIDGET LN,COLFAX,CA,95713,"U.S. Bank, National Association",6548.0,...,1.0,INDIVIDUAL,Unanswered,P I F,2024-11-30,0.0,N,0.0,Y,NaN
3,2026-06-30,7A,123499,"513 SOLUTIONS GROUP, LLC",120 FIREBIRD RUN,CIBOLO,TX,78108,BayFirst National Bank,34997.0,...,15.0,CORPORATION,Existing or more than 2 years old,EXEMPT,NaT,0.0,N,3.0,N,Y
4,2026-06-30,7A,70400,"Tian Shan International, LLC",2503 BAGBY ST,HOUSTON,TX,77006,Golden Bank National Association,26223.0,...,2.0,CORPORATION,Unanswered,P I F,2024-01-31,0.0,N,4.0,Y,NaN


## 4. Prepare the cleaned SBA dataset

The cleaned dataframe is now written to the project's prepared-data directory using the shared `prepare_dataset` workflow. This keeps the raw staging files and the processed artifact in the repo's standard locations and records the data dictionary alongside the prepared output.

The task is defined as classification, consistent with the repo's other dataset preparation notebooks.


In [2]:
from pathlib import Path
import yaml
import pandas as pd

from data_prep.config import bootstrap_config, load_config
from data_prep.prepare import prepare_dataset


def prepare_sba_dataset(dataframe, *, prepared_dataset_dir, dataset_name="sba", output_format="csv", metadata=None):
    """Notebook-local SBA preparation logic."""
    cleaned = dataframe.copy()
    if cleaned.empty:
        raise ValueError("SBA dataset is empty after cleaning.")

    result_metadata = {
        "source": "sba",
        "preparation_logic": "drop_duplicates_missing_normalize_columns",
    }
    if metadata:
        result_metadata.update(metadata)

    return prepare_dataset(
        dataset_name=dataset_name,
        representation="tabular",
        task_characterization="classification",
        staging_dir="./data/raw/sba",
        prepared_dataset_dir=prepared_dataset_dir,
        data=cleaned,
        output_format=output_format,
        filename=f"{dataset_name}_prepared",
        metadata=result_metadata,
    )


project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

prepared_dir = project_root / "data" / "prepared" / "sba"
prepared_dir.mkdir(parents=True, exist_ok=True)

config_path = bootstrap_config(
    config_dir=project_root / "configs",
    config_name="sba-prep.yaml",
    dataset_name="sba",
    representation="tabular",
    task_characterization="classification",
    staging_dir="./data/raw/sba",
    prepared_dataset_dir=str(prepared_dir),
    write=True,
)

raw_dd = pd.read_csv(data_dict_path)
if "attribute" in raw_dd.columns:
    dict_key_col = "attribute"
else:
    dict_key_col = raw_dd.columns[0]

normalized_cols = {str(col).strip().lower() for col in df.columns}
processed_dd = raw_dd[
    raw_dd[dict_key_col].astype(str).str.strip().str.lower().isin(normalized_cols)
].copy().reset_index(drop=True)

prepare_result = prepare_sba_dataset(
    df,
    prepared_dataset_dir=prepared_dir,
    dataset_name="sba",
    output_format="csv",
    metadata={
        "raw_data_dictionary": raw_dd,
        "data_dictionary": processed_dd,
    },
)

config = load_config(config_path)
config["raw_data_dictionary_path"] = str(data_dict_path)
config["data_dictionary_path"] = str(Path(prepare_result["dictionary_files"]["data_dictionary"]))
config["processed_data_dictionary_path"] = config["data_dictionary_path"]
config["prepared_dataset_dir"] = str(prepared_dir)
with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f"Bootstrap config: {config_path}")
print(f"Prepared dataset: {prepare_result['output_path']}")
print(f"Dictionary files: {prepare_result.get('dictionary_files', {})}")
print(f"Files in prepared folder: {sorted(p.name for p in prepared_dir.iterdir())}")

pd.read_csv(prepare_result['output_path']).head()


Bootstrap config: /home/rajiv/programming/kmds-dataset-util/configs/sba-prep.yaml
Prepared dataset: /home/rajiv/programming/kmds-dataset-util/data/prepared/sba/sba_prepared.csv
Dictionary files: {'data_dictionary': 'data/raw/sba/data_dictionary_processed.csv', 'raw_data_dictionary': 'data/raw/sba/data_dictionary_raw.csv'}
Files in prepared folder: ['sba_prepared.csv']


,asofdate,program,locationid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,congressionaldistrict,businesstype,businessage,loanstatus,paidinfulldate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,2026-06-30,7A,29805,"AMERIPRO CONSTRUCTION SERVICES, INC.",1403 SENTRY LANE,Norristown,PA,19403,"TD Bank, National Association",18409.0,...,4.0,PARTNERSHIP,Unanswered,P I F,2024-11-30,0.0,Y,5.0,N,NaN
1,2026-06-30,7A,84894,Meluota Corp,2702 ASTORIA BLVD,ASTORIA,NY,11102,"Santander Bank, National Association",29950.0,...,10.0,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,0.0,Y,3.0,N,NaN
2,2026-06-30,7A,53803,THOMAS W CHASE,1700 GIDGET LN,COLFAX,CA,95713,"U.S. Bank, National Association",6548.0,...,1.0,INDIVIDUAL,Unanswered,P I F,2024-11-30,0.0,N,0.0,Y,NaN
3,2026-06-30,7A,123499,"513 SOLUTIONS GROUP, LLC",120 FIREBIRD RUN,CIBOLO,TX,78108,BayFirst National Bank,34997.0,...,15.0,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,0.0,N,3.0,N,Y
4,2026-06-30,7A,70400,"Tian Shan International, LLC",2503 BAGBY ST,HOUSTON,TX,77006,Golden Bank National Association,26223.0,...,2.0,CORPORATION,Unanswered,P I F,2024-01-31,0.0,N,4.0,Y,NaN
